# Part 2: Python Port Scanner Development

## TCP Scanner

In [ ]:
import socket
import sys

def tcp_scanner(target, port):
   """
   # 1. Add your comment here (e.g., Why do we use `socket.AF_INET` and `socket.SOCK_STREAM`?)
   socket.AF_INET means that we are using IPv4 addresses which is why it is chosen
   socket.SOCK_STREAM creates a reliable TCP protocol which is used for scanning TCP ports
   together we are able to create an IPv4 TCP socket.
   """
   try:
         tcp_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
         """
         # 2. What does settimeout(1) do for the TCP socket?
         we need a check against a filtered port or it could block indefinitely
         a 1sec timeout prevents our script from hanging on non-responsive ports
         """

         tcp_sock.settimeout(1)
         tcp_sock.connect((target, port))
         tcp_sock.close()
         return True
   except:
         """
         # 3. Add your comment here (e.g., What types of exceptions might occur?)
         we have an exceptions to catch everything that is not an open port
         this could be closed ports, filtered ports, timeout errors, bad addresses, etc.
         """

         """
         # 4. Why is it important to handle exceptions in network programming?
         exception handling is important in any programming but especially so in network programming
         as networks can be unstable and unreliable so we need to be able to handle any exceptions
         so that our script does not stall on the first thing that goes wrong
         we want it to continue and skip an issues possible.
         """
         return False

def main():
   """
   # 5. Add your comment here (e.g., Why check for command-line arguments with `len(sys.argv)`?)
   it makes sure that the script is only ran if both the script name and the target IP are provided as arguments
   the script is written to only fuction if 2 arguments are provided
   """

   """
   # 6. What happens if no arguments are passed to the script?
   if would then print the usage of the tcp_scanner.py on the IP and exit, this form of error handling
   is preferred as it allows us to easily see what failed and where
   """
   if len(sys.argv) != 2:
         print("Usage: python tcp_scanner.py <Metasploitable-2_IP>")
         sys.exit(1)

   target = sys.argv[1]
   print(f"Scanning TCP ports on {target}...")
   """
   # 7. Add your comment here (e.g., Why loop through the port range 1-1024?)
   common well-known ports are 0-1023, 1-1024 is specified since 0 is reserved on non functional
   up to 1024 is used since python range is exclusive of the end value and we want to end on 1023
   """
   for port in range(1, 1024):
         if tcp_scanner(target, port):
            print(f"[*] Port {port}/tcp is open")

if __name__ == "__main__":
   main()

   """
   # 8. Does running this script require sudo? Why or why not?

   i do not believe this script would require sudo as we are using TCP connect through a socket
   so this should not require special admin privileges to run, it is just a scanning script
   """

### Part 2: Task 1

1. Why do we need to create a socket using socket.AF_INET and socket.SOCK_STREAM?
   socket.AF_INET means that we are using IPv4 addresses which is why it is chosen socket.SOCK_STREAM creates a reliable TCP protocol which is used for scanning TCP ports together we are able to create an IPv4 TCP socket.
2. What does settimeout(1) do for the TCP socket?
   We need a check against a filtered port or it could block indefinitely a 1sec timeout prevents our script from hanging on non-responsive ports.
3. What types of exceptions might occur in the except block?
   We have an exceptions to catch everything that is not an open port this could be closed ports, filtered ports, timeout errors, bad addresses, etc.
4. Why is it important to handle exceptions in network programming?
   Exception handling is important in any programming but especially so in network programming as networks can be unstable and unreliable so we need to be able to handle any exceptions so that our script does not stall on the first thing that goes wrong we want it to continue and skip an issues possible.
5. Why do we check for command-line arguments using len(sys.argv)?
   It makes sure that the script is only ran if both the script name and the target IP are provided as arguments the script is written to only fuction if 2 arguments are provided
6. What happens if no arguments are passed to the script?
   If would then print the usage of the tcp_scanner.py on the IP and exit, this form of error handling is preferred as it allows us to easily see what failed and where.
7. Why do we loop through the port range (1, 1024)? What are these ports called?
   Common well-known ports are 0-1023, 1-1024 is specified since 0 is reserved on non functional up to 1024 is used since python range is exclusive of the end value and we want to end on 1023.
8. Does running this script require sudo? Why or why not?
   I do not believe this script would require sudo as we are using TCP connect through a socket so this should not require special admin privileges to run, it is just a scanning script

meta VM: 192.168.10.103

created a shared folder with the KaliAttack VM
ls /media
cd /media/sf_KaliAttack_Shared
ls
	tcp_scanner.py   # search for the script in the shared folder
cp /media/sf_KaliAttack_Shared/tcp_scanner.py ~/   # copy the script onto the home directory
ls ~
	tcp_scanner.py   # the script is here so we are good to run our script
python3 tcp_scanner.py 192.168.10.103
	Scanning TCP ports on 192.168.10.103...
	[*] Port 21/tcp is open
	[*] Port 22/tcp is open
	[*] Port 23/tcp is open
	[*] Port 25/tcp is open
	[*] Port 53/tcp is open
	[*] Port 80/tcp is open
	[*] Port 111/tcp is open
	[*] Port 139/tcp is open
	[*] Port 445/tcp is open
	[*] Port 512/tcp is open
	[*] Port 513/tcp is open
	[*] Port 514/tcp is open


## UDP DNS Port Scanner

In [ ]:
import socket
import sys

def udp_dns_scanner(target, port=53):
   """
   # Add your comment here (e.g., Why use `socket.SOCK_DGRAM` for UDP scanning?)
   """
   try:
         udp_sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
         udp_sock.settimeout(2.0)  # Add your comment here (e.g., What does `settimeout` do?)

         query = b'\x12\x34\x01\x00\x00\x01\x00\x00\x00\x00\x00\x00' \
               b'\x07example\x03com\x00\x00\x01\x00\x01'
         # Add your comment here (e.g., What is the purpose of the `query` variable?)

         udp_sock.sendto(query, (target, port))

         response, _ = udp_sock.recvfrom(512)
         # Add your comment here (e.g., Why use `recvfrom(512)`? What does 512 represent?)

         if response:
            print(f"[*] Port {port}/udp (DNS) is open and responding")
            return True
   except socket.timeout:
         # Add your comment here (e.g., Why is a timeout exception likely in UDP scanning?)
         print(f"[-] Port {port}/udp (DNS) did not respond")
         return False
   finally:
         # Add your comment here (e.g., Why close the socket in the `finally` block?)
         udp_sock.close()

def main():
   """
   # Add your comment here (e.g., What is the purpose of `sys.argv`?)
   """
   if len(sys.argv) != 2:
         print("Usage: python udp_dns_scanner.py <Metasploitable-2_IP>")
         sys.exit(1)

   target = sys.argv[1]
   print(f"Scanning UDP DNS port on {target}...")
   udp_dns_scanner(target)

if __name__ == "__main__":
   main()

### Part 2: Task 2

1. Why do we use socket.SOCK_DGRAM for UDP scanning?
   In this script we are using socket.SOCK_DGRAM as oppose to the SOCK_STREAM because DGRAM is for UDP communicationn while STREAM is for TCP communication, this is a UDP Port Scannner so thus we use DGRAM
2. What does the timeout argument achieve for the socket?
   As like in the previous TCP script we add a timeout to prevent our script from stalling from it being closed or filtered, as we may recieve no response. Since UDP has no handshake our script could stall indefinitely which is why we have the 2sec timeout.
3. What is the purpose of the query variable, and why does it contain these specific bytes?
   UDP cannot use connection to define a successful connection we need a way to do so. That is what the purpose of the query is which defines a raw DNS query packet in bytes. We evaluate the scanning by sending a formatted DNS request and getting a response.
4. Why do we use recvfrom(512) to read responses? What does 512 represent?
   512 bytes is used as the size for the size of the response because this is the maximum size of DNS or UDP. So if we get a response we want to make sure we capture the entirety of the response data.
5. Why is a timeout exception likely in UDP scanning?
   In UDP scanning timeout is a much greater concern since due to security there is a lot more ways to be left stalling since closed UDP ports slielntly drop packets, UDP does not send back RST like it would in TCP, and Firewalls will drop UDP probes.
6. Why is it important to close the socket in the finally block?
   Unlike connect() in TCP which explictly closes the socket after use, UDP allowd multipe operations to happen within a socket creation thus requires it to be manually closed once finished. It is important to close since keeping this socket open is resource intensive.
7. What is the purpose of sys.argv in the main function?
   Like in the TCP scanner we want to use sys.argv in this context to only allow the script to run if the correct syntax and agruments are provided in order for the script to run successfully, we should be applying error handling anywhere applicable to avoid and strange outcomes.
8. Why is input validation important in this case?
   When we have an error in input we want a clear defined output informing the user (us) what went wrong for clarity. Adding input validation is as simple way to ensure that this portion is covered and easily debugged.
9.  Does running this script require sudo? Why or why not?
    This script does not require sudo either since we are using standard UDP send/receive operations within the network's standard operations. There is no need for elevated privledges since low-level networking isnt used nor kernal functionality.

ls /media
cd /media/sf_KaliAttack_Shared
ls
	udp_dns__scanner.py   # search for the script in the shared folder
cp /media/sf_KaliAttack_Shared/udp_dns__scanner.py ~/   # copy the script onto the home directory
ls ~
	tcp_scanner.py   # the script is here so we are good to run our script
python3 udp_dns__scanner.py 192.168.10.103
	Scanning UDP DNS port on 192.168.10.103...
	[*] Port 53/udp (DNS) is open and responding

### Summary and Analysis Report

There were several things of note in using these two scanning scripts, it was an interesting processes overall. What was obvious was the need to be  very precise in how the script was written as to how it would handle errors of things that were not expected. Since we are scanning a potentially unknown environment we need to account for all the possibiltles we can so that we are able to preform a successful scan or get a clean output as to what went wrong. That is why we need things such as timeouts and sys.argv to ensure that the script does not timeout. The difference between TCP and UDP scanning was also neat, as they were both on standard communication so there was no need for sudo. They both used sockets but different types, SOCK_STREAM for TCP communication vs SOCK_DGRAM for UDP communication. There are also differences in how we decide to preform the scan, such as for TCP we search all common well-known ports to see which are open while in UDP we only hit port 53 since that is the DNS port that we wanted to specifically hit. TCP closed the port after we interact with it while UDP requires us to manually close the port since it allows multiple operations while the socket is open.

UDP Port scanning is tougher than TCP Port Scanning for several reasons. For one UDP has more features that make it slightly more secure than TCP by default so we need to work around them. This is such as UDP does not require a handshake so we need another form to check the port so that is why we use the query variable to check if sending it a formatted DNS request results in a reponse. In addition UDP ports are known to silently drop packets for a variety of reasons so it is important to design it in a way that avoids the many ways that the script could stall out.